# 9 — Positions vs Graph Structure: linking learned attention to trading decisions

Closes the §5d gap in [04_interpretability_plan.md](../notes/04_interpretability_plan.md). Tests whether GAT is simply 'GCN scaled down' (`W_msg` magnitude effect) or whether its learned graph structure actually drives different trading decisions — carefully avoiding flattened measurements.

Sections:
1. Load & align positions
2. Per-stock TIME-SERIES position correlation (not cross-sectional, not flattened)
3. Scale-vs-direction decomposition: α_i and residual variance fraction
4. Turnover per model
5. Per-stock 2020 and 2022 trade divergence
6. **Core test**: per-stock attention divergence (from NB8) vs position residual
   — per-stock, per-stock×per-window within-stock, regime split, density partial
7. Summary verdicts

**Requires notebook 8 to have run** (reads `notebooks/8_results/per_stock_row_divergence.csv`).


## 0. Setup


In [ ]:
import os, sys
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        get_ipython().system('git clone https://github.com/adam-909/4yp.git /content/repo')
    else:
        get_ipython().system('cd /content/repo && git pull')
    os.chdir('/content/repo/4YP-main')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
    from google.colab import drive
    drive.mount('/content/drive')
elif os.path.exists("/mnt/g/My Drive"):
    os.chdir('/home/adam/new4YP/4YP-main')
    RESULTS_BASE = "/mnt/g/My Drive/FINAL_RESULTS"
else:
    os.chdir('/home/adam/new4YP/4YP-main')
    RESULTS_BASE = "FINAL_RESULTS"
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd(), "| RESULTS_BASE:", RESULTS_BASE)

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from scipy.spatial.distance import jensenshannon
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

from gml.experiment_utils import load_experiment_results
from settings.default import ALL_TICKERS, BBG_SECTORS

plt.rcParams.update({"font.size": 11, "figure.dpi": 120,
                     "axes.grid": True, "grid.alpha": 0.3, "figure.facecolor": "white"})

In [ ]:
GCN_EXPERIMENT = "3_GCN_rolling_pearson"; GCN_CONFIG = "lb20_th0.4_equity"; GCN_SEED = 42
GAT_EXPERIMENT = "4c_GATv2_rolling_pearson_th0.4_scFalse_resTrue"; GAT_CONFIG = "lb20_th0.4_equity"; GAT_SEED = 40

gcn = load_experiment_results(GCN_EXPERIMENT, GCN_SEED, config_name=GCN_CONFIG, base_dir=RESULTS_BASE)
gat = load_experiment_results(GAT_EXPERIMENT, GAT_SEED, config_name=GAT_CONFIG, base_dir=RESULTS_BASE)

pearson   = gcn["adjacency"]                      # (W, 88, 88) rolling Pearson weights (with mask applied)
gat_attn  = gat["attention_weights"].mean(axis=1) # (W, 88, 88) head-averaged
dates     = pd.to_datetime(gat["test_dates"])
tickers   = list(gcn.get("tickers", ALL_TICKERS))[:gat_attn.shape[-1]]
sectors   = pd.Series(BBG_SECTORS).reindex(tickers)

mask = (pearson != 0)  # shared Pearson mask
print("pearson:", pearson.shape, "| gat_attn:", gat_attn.shape)
print("mean edges per window:", mask.sum(axis=(1,2)).mean())
gcn_cr = gcn["captured_returns"]; gat_cr = gat["captured_returns"]

def pivot_positions(cr_df):
    d = cr_df.copy(); d["time"] = pd.to_datetime(d["time"])
    return d.pivot_table(index="time", columns="identifier", values="position", aggfunc="first")

gcn_pos = pivot_positions(gcn_cr); gat_pos = pivot_positions(gat_cr)
common_dates = gcn_pos.index.intersection(gat_pos.index)
common_tickers = [t for t in tickers if t in gcn_pos.columns and t in gat_pos.columns]
gcn_pos = gcn_pos.loc[common_dates, common_tickers]
gat_pos = gat_pos.loc[common_dates, common_tickers]
sectors_c = pd.Series(BBG_SECTORS).reindex(common_tickers)
print("aligned positions:", gcn_pos.shape)

OUT_DIR_9 = "notebooks/9_results"
os.makedirs(OUT_DIR_9, exist_ok=True)

## 9.2 Per-stock time-series position correlation


In [ ]:
# 9.2: Per-stock time-series position correlation (NOT cross-sectional, NOT flattened)
rows = []
for t in common_tickers:
    g = gcn_pos[t]; a = gat_pos[t]
    valid = g.notna() & a.notna()
    if valid.sum() < 50: continue
    r = stats.pearsonr(g[valid], a[valid])[0]
    rs = stats.spearmanr(g[valid], a[valid])[0]
    rows.append((t, sectors_c[t], r, rs, g.std(), a.std()))
per_stock_rho = pd.DataFrame(rows, columns=["ticker","sector","rho","rho_spearman","std_gcn","std_gat"])
print(per_stock_rho["rho"].describe())

fig, ax = plt.subplots(1, 2, figsize=(14,4))
ax[0].hist(per_stock_rho["rho"], bins=25); ax[0].set_title("Per-stock time-series position ρ (GAT vs GCN)")
ax[0].axvline(per_stock_rho["rho"].median(), color="r", ls="--", label=f"median {per_stock_rho['rho'].median():.2f}")
ax[0].legend()
sns.boxplot(data=per_stock_rho, x="sector", y="rho", ax=ax[1])
ax[1].set_title("By sector"); ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()
per_stock_rho.to_csv(f"{OUT_DIR_9}/per_stock_position_correlation.csv", index=False)

# By year
years = sorted(common_dates.year.unique())
year_rho = pd.DataFrame(index=common_tickers, columns=years, dtype=float)
for y in years:
    mask_y = common_dates.year == y
    for t in common_tickers:
        g = gcn_pos.loc[mask_y, t]; a = gat_pos.loc[mask_y, t]
        v = g.notna() & a.notna()
        if v.sum() > 30:
            year_rho.loc[t, y] = stats.pearsonr(g[v], a[v])[0]
fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(year_rho.astype(float), cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label":"ρ"})
ax.set_title("Per-stock × per-year position correlation (GAT vs GCN)")
plt.tight_layout(); plt.show()
year_rho.to_csv(f"{OUT_DIR_9}/per_stock_year_rho.csv")

## 9.3 Scale-vs-direction decomposition


In [ ]:
# 9.3: Scale-vs-direction decomposition — α_i s.t. α_i · gcn ≈ gat
alpha = {}; resid_var_frac = {}
for t in common_tickers:
    g = gcn_pos[t].values; a = gat_pos[t].values
    v = ~np.isnan(g) & ~np.isnan(a)
    if v.sum() < 50: continue
    gv, av = g[v], a[v]
    denom = (gv @ gv)
    if denom < 1e-12: continue
    ai = (gv @ av) / denom
    alpha[t] = ai
    resid = av - ai * gv
    resid_var_frac[t] = resid.var() / (av.var() + 1e-12)
dec = pd.DataFrame({"alpha": alpha, "resid_var_frac": resid_var_frac})
dec["sector"] = sectors_c.reindex(dec.index)
print(dec.describe())
print(f"\nalpha median {dec['alpha'].median():.3f}  (plan expects 0.5-0.7)")
print(f"resid_var_frac median {dec['resid_var_frac'].median():.3f}  (low ⇒ scaled-down story; high ⇒ directional differences)")

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].hist(dec["alpha"], bins=25); ax[0].axvline(1.0, color="k", ls=":")
ax[0].axvline(dec["alpha"].median(), color="r", ls="--", label=f"median {dec['alpha'].median():.2f}")
ax[0].set_title("Per-stock scale α_i"); ax[0].legend()
ax[1].hist(dec["resid_var_frac"], bins=25)
ax[1].set_title("Residual variance fraction after scale removal")
plt.tight_layout(); plt.show()
dec.to_csv(f"{OUT_DIR_9}/scale_decomposition.csv")

## 9.4 Turnover per model


In [ ]:
# 9.4: Turnover per model
def turnover(pos):
    return pos.diff().abs()

to_gcn = turnover(gcn_pos); to_gat = turnover(gat_pos)
total_to = pd.DataFrame({"GCN": to_gcn.sum(axis=1), "GAT": to_gat.sum(axis=1)})
per_stock_to = pd.DataFrame({"GCN": to_gcn.mean(), "GAT": to_gat.mean()}).assign(sector=sectors_c)

print("Daily turnover (sum |Δp|):"); print(total_to.mean())
print("\nPer-stock mean daily turnover:"); print(per_stock_to.describe())

fig, ax = plt.subplots(1, 2, figsize=(14,4))
ax[0].plot(common_dates, total_to["GCN"].rolling(20).mean(), label="GCN")
ax[0].plot(common_dates, total_to["GAT"].rolling(20).mean(), label="GAT")
ax[0].set_title("Total daily turnover (20d MA)"); ax[0].legend(); ax[0].tick_params(axis="x", rotation=30)
ax[1].scatter(per_stock_to["GCN"], per_stock_to["GAT"], s=20, alpha=0.7)
lim = max(per_stock_to["GCN"].max(), per_stock_to["GAT"].max())
ax[1].plot([0,lim],[0,lim], "k--", alpha=0.4)
ax[1].set_xlabel("GCN per-stock turnover"); ax[1].set_ylabel("GAT per-stock turnover")
ax[1].set_title("Per-stock turnover parity check")
plt.tight_layout(); plt.show()
total_to.to_csv(f"{OUT_DIR_9}/total_turnover.csv")
per_stock_to.to_csv(f"{OUT_DIR_9}/per_stock_turnover.csv")

## 9.5 Per-stock 2020 and 2022 divergence


In [ ]:
# 9.5: Per-stock 2020 and 2022 trade divergence
def divergence_year(year):
    msk = common_dates.year == year
    dates_y = common_dates[msk]
    rows = []
    for t in common_tickers:
        g = gcn_pos.loc[msk, t].values; a = gat_pos.loc[msk, t].values
        v = ~np.isnan(g) & ~np.isnan(a)
        if v.sum() < 30: continue
        gv, av = g[v], a[v]
        denom = (gv @ gv)
        ai = (gv @ av) / denom if denom > 1e-12 else np.nan
        resid_rms = np.sqrt(np.mean((av - (ai if np.isfinite(ai) else 1.0) * gv) ** 2))
        rows.append((t, sectors_c[t], ai, resid_rms,
                     stats.pearsonr(gv, av)[0] if gv.std() > 1e-12 and av.std() > 1e-12 else np.nan))
    return pd.DataFrame(rows, columns=["ticker","sector","alpha","resid_rms","rho"])

div_2020 = divergence_year(2020).sort_values("resid_rms", ascending=False)
div_2022 = divergence_year(2022).sort_values("resid_rms", ascending=False)
print("Top-15 stocks GAT traded differently in 2020 (COVID):")
print(div_2020.head(15).to_string(index=False))
print("\nTop-15 stocks GAT traded differently in 2022 (rate hikes):")
print(div_2022.head(15).to_string(index=False))

# Sector concentration of top-divergent stocks
for label, df in [("2020", div_2020), ("2022", div_2022)]:
    top = df.head(20)
    print(f"\n{label} top-20 sector mix:"); print(top["sector"].value_counts())

div_2020.to_csv(f"{OUT_DIR_9}/divergence_2020.csv", index=False)
div_2022.to_csv(f"{OUT_DIR_9}/divergence_2022.csv", index=False)

# Plot 4 most-divergent stocks for each regime
for label, df in [("2020", div_2020), ("2022", div_2022)]:
    top4 = df.head(4)["ticker"].tolist()
    msk = common_dates.year == int(label)
    fig, axes = plt.subplots(2, 2, figsize=(12,6), sharex=True)
    for ax_, t in zip(axes.flat, top4):
        ax_.plot(common_dates[msk], gcn_pos.loc[msk, t], label="GCN")
        ax_.plot(common_dates[msk], gat_pos.loc[msk, t], label="GAT", alpha=0.8)
        ax_.set_title(f"{t} ({sectors_c[t]}) — {label}"); ax_.legend(fontsize=8)
        ax_.tick_params(axis="x", rotation=30)
    plt.tight_layout(); plt.show()

## 9.6 Linking graph structure to position differences
The key new analysis. Three nested tests:
- Per-stock: attention-divergence (mean JS from NB8) vs position residual variance
- Per-stock × per-day, within-stock (no flattening across stocks): row-JS vs |position residual|
- Regime split + partial correlation controlling for graph density


In [ ]:
# 9.6: THE KEY TEST — does per-stock position divergence track per-stock attention divergence?
# Load §8.3 per-stock row-divergence output from notebook 8.
try:
    row_div = pd.read_csv(f"notebooks/8_results/per_stock_row_divergence.csv")
except FileNotFoundError:
    print("Run notebook 8 first (needs per_stock_row_divergence.csv)")
    raise

link = row_div[["ticker","sector","mean_JS"]].merge(
    dec.reset_index().rename(columns={"index":"ticker"})[["ticker","alpha","resid_var_frac"]],
    on="ticker", how="inner")
print(link.describe())

# Test 1: per-stock correlation (attention-JS vs position-residual-var-frac)
r, p = stats.pearsonr(link["mean_JS"], link["resid_var_frac"])
rs, ps = stats.spearmanr(link["mean_JS"], link["resid_var_frac"])
print(f"\nPer-stock: Pearson r={r:.3f} (p={p:.4f}) | Spearman ρ={rs:.3f} (p={ps:.4f})")
print("If significant positive: GAT's attention divergence from Pearson partly explains its position divergence.")

fig, ax = plt.subplots(figsize=(7,5))
for sec, grp in link.groupby("sector"):
    ax.scatter(grp["mean_JS"], grp["resid_var_frac"], label=sec, s=25, alpha=0.75)
z = np.polyfit(link["mean_JS"], link["resid_var_frac"], 1)
x = np.linspace(link["mean_JS"].min(), link["mean_JS"].max(), 100)
ax.plot(x, np.polyval(z, x), "k--", alpha=0.6, label=f"fit r={r:.2f}")
ax.set_xlabel("Mean row-wise JS (GAT vs Pearson)")
ax.set_ylabel("Residual variance fraction (position, post scale-removal)")
ax.set_title("Per-stock: attention divergence vs position-direction divergence")
ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()
link.to_csv(f"{OUT_DIR_9}/link_attention_to_position.csv", index=False)

In [ ]:
# 9.6 continued: per-stock × per-window within-stock test (removes stock fixed effects)
# For each (i, t) compute: row_js(i,t) and position residual |gat_pos - α_i · gcn_pos|.
# Then within each stock, correlate the two time series. Avoids flattening.
#
# Needs row_js array from notebook 8. Recompute locally (cheap).
eps = 1e-12
pearson_abs = np.abs(pearson)
pearson_p = pearson_abs / (pearson_abs.sum(axis=2, keepdims=True) + eps)
gat_p = gat_attn / (gat_attn.sum(axis=2, keepdims=True) + eps)

# Window dates — align with common_dates by forward-fill of the window date → daily
win_dates = pd.to_datetime(gat["test_dates"])
# Map each daily date to its window index (nearest earlier window)
win_of_day = np.searchsorted(win_dates.values, common_dates.values, side="right") - 1
win_of_day = np.clip(win_of_day, 0, len(win_dates) - 1)

W, N, _ = gat_attn.shape
# ticker → column index alignment
ticker_idx = {t: i for i, t in enumerate(tickers)}

within_rows = []
for t in common_tickers:
    if t not in ticker_idx: continue
    i = ticker_idx[t]
    # Per-day row JS (use window for that day)
    row_js_day = np.array([jensenshannon(pearson_p[w, i], gat_p[w, i]) for w in win_of_day])
    # Position residual after scale removal
    g = gcn_pos[t].values; a = gat_pos[t].values
    v = ~np.isnan(g) & ~np.isnan(a) & ~np.isnan(row_js_day)
    if v.sum() < 80: continue
    ai = alpha.get(t, np.nan)
    if not np.isfinite(ai): continue
    resid_abs = np.abs(a[v] - ai * g[v])
    js_v = row_js_day[v]
    if js_v.std() < 1e-8 or resid_abs.std() < 1e-8: continue
    r_ws = stats.pearsonr(js_v, resid_abs)[0]
    within_rows.append((t, sectors_c[t], r_ws, v.sum()))
within_df = pd.DataFrame(within_rows, columns=["ticker","sector","within_r","n"])
print("Within-stock per-day r (JS vs |position residual|):")
print(within_df["within_r"].describe())
pos = (within_df["within_r"] > 0).mean()
t_stat, p_val = stats.ttest_1samp(within_df["within_r"].dropna(), 0.0)
print(f"Fraction with r>0: {pos:.2%}  |  t={t_stat:.2f}, p={p_val:.4f}")

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(within_df["within_r"], bins=25); ax.axvline(0, color="k")
ax.axvline(within_df["within_r"].median(), color="r", ls="--", label=f"median {within_df['within_r'].median():.3f}")
ax.set_title("Within-stock correlation: row-JS vs position residual (88 stocks)")
ax.legend(); plt.tight_layout(); plt.show()
within_df.to_csv(f"{OUT_DIR_9}/within_stock_link.csv", index=False)

In [ ]:
# 9.6 regime split + density control
# VIX regimes already defined if loaded; reload here to be self-contained.
import yfinance as yf
vix = yf.download("^VIX", start="2017-01-01", end="2023-09-01")["Close"].squeeze()
vix.index = pd.to_datetime(vix.index)
vix_day = vix.reindex(common_dates, method="ffill")
stress_day = (vix_day > vix_day.quantile(0.75)).values

# Density per day = mask row-sum
density_day = np.array([mask[w].sum(axis=1).mean() for w in win_of_day])

regime_rows = []
for t in common_tickers:
    if t not in ticker_idx: continue
    i = ticker_idx[t]
    row_js_day = np.array([jensenshannon(pearson_p[w, i], gat_p[w, i]) for w in win_of_day])
    g = gcn_pos[t].values; a = gat_pos[t].values
    ai = alpha.get(t, np.nan)
    if not np.isfinite(ai): continue
    resid_abs = np.abs(a - ai * g)
    for regime, msk_r in [("stable", ~stress_day), ("stress", stress_day)]:
        v = msk_r & ~np.isnan(row_js_day) & ~np.isnan(resid_abs)
        if v.sum() < 30: continue
        if row_js_day[v].std() < 1e-8 or resid_abs[v].std() < 1e-8: continue
        r_r = stats.pearsonr(row_js_day[v], resid_abs[v])[0]
        # Partial correlation controlling for density (residualise both on density)
        d = density_day[v]; js_r = row_js_day[v]; rd_r = resid_abs[v]
        def resid_of(y):
            X = np.c_[np.ones_like(d), d]
            coef, *_ = np.linalg.lstsq(X, y, rcond=None); return y - X @ coef
        js_c = resid_of(js_r); rd_c = resid_of(rd_r)
        r_partial = stats.pearsonr(js_c, rd_c)[0] if js_c.std() > 1e-8 and rd_c.std() > 1e-8 else np.nan
        regime_rows.append((t, sectors_c[t], regime, r_r, r_partial, v.sum()))
regime_df = pd.DataFrame(regime_rows, columns=["ticker","sector","regime","r","r_partial_density","n"])
print("Within-stock correlation by regime (mean across stocks):")
print(regime_df.groupby("regime")[["r","r_partial_density"]].agg(["mean","median","count"]))
regime_df.to_csv(f"{OUT_DIR_9}/within_stock_link_by_regime.csv", index=False)

fig, ax = plt.subplots(figsize=(8,4))
sns.boxplot(data=regime_df, x="regime", y="r_partial_density", ax=ax)
ax.axhline(0, color="k", lw=0.5); ax.set_title("Partial within-stock r (controlling for graph density)")
plt.tight_layout(); plt.show()

## 9.7 Verdict


In [ ]:
# 9.7: Summary verdicts
def verdict(row):
    if row["resid_var_frac"] < 0.15 and row["alpha"] > 0: return "scale-dominated"
    if row["resid_var_frac"] > 0.40: return "structure-dominated"
    return "mixed"
dec["verdict"] = dec.apply(verdict, axis=1)
print("Verdict counts:"); print(dec["verdict"].value_counts())
print("\nAlpha by verdict:"); print(dec.groupby("verdict")["alpha"].describe())
dec.to_csv(f"{OUT_DIR_9}/scale_decomposition_with_verdict.csv")

print("\nNotebook 9 complete. Artifacts under", OUT_DIR_9)